In [3]:
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
import wandb

MODEL_NAME = 'Qwen/Qwen3-4B-Instruct'
DATA_PATH = 'data/insta-sft.jsonl'
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

NotImplementedError: Unsloth cannot find any torch accelerator? You need a GPU.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    lora_dropout=0,
    target_modules=[
        'q_proj','k_proj','v_proj','o_proj',
        'gate_proj','up_proj','down_proj'
    ],
)

In [ ]:
dataset = load_dataset('json', data_files=DATA_PATH, split='train')
dataset

In [ ]:
wandb.init(project='gur-prime')

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        output_dir='outputs/checkpoints',
        num_train_epochs=1,
        learning_rate=2e-4,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        logging_steps=10,
        save_steps=200,
        max_seq_length=2048,
        bf16=False,
        fp16=True,
        packing=True,
        report_to='wandb',
    ),
)

trainer.train()

In [ ]:
model.save_pretrained('outputs/adapter')
tokenizer.save_pretrained('outputs/adapter')